In [2]:
### 7-2-1. Text Classification with HuggingFace Transformers

!pip install transformers # 허깅페이스 transformers 라이브러리 다운로드

from transformers import BertTokenizer, BertForSequenceClassification
import torch

# 사전 학습된 영어 BERT 모델과 토크나이저 호출
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased',
                                                      num_labels=2)

# 예제 문장(감성 분석: 긍정/부정)
sentence = "This movie was fantastic!"
inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)

# 모델 예측
outputs = model(**inputs) # ** → 파이썬의 Dictionary Unpacking 연산자
logits = outputs.logits
'''
각 데이터별로 여러 클래스 중 어떤 클래스의 점수가 가장 높은지 비교해야 한다
따라서 dim=1
'''
predicted_class = torch.argmax(logits, dim=1)
print(f"예측된 감성 클래스: {predicted_class.item()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


예측된 감성 클래스: 1


In [3]:
import urllib.request

# 테스트용 고양이 이미지 다운로드 URL for code 7-2-2
url = "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"
save_path = "example.jpg"

# 403 Forbidden 우회를 위해 User-Agent 헤더 추가
req = urllib.request.Request(
    url,
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
)

try:
    with urllib.request.urlopen(req) as response, open(save_path, 'wb') as out_file:
        data = response.read()
        out_file.write(data)
    print("테스트 이미지(example.jpg) 다운로드 성공!")
except Exception as e:
    print(f"다운로드 실패: {e}")

테스트 이미지(example.jpg) 다운로드 성공!


In [4]:
### 7-2-2. Image Classifications and Transfer-Learning with Pre-trained ResNet

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image


## 사전 학습된 ResNet-18 모델 불러오기
resnet18 = models.resnet18(pretrained=True)

## 기존 분류기를 새로운 데이터셋에 맞게 교체(예: 10개 클래스 분류)

# remind → 2026.07.26 당시 Oxford-IIIT Pet Transfer Learning에서 사용 경험
num_ftrs = resnet18.fc.in_features # number of features(특징의 개수)
resnet18.fc = nn.Linear(num_ftrs, 10)


## 모델 평가 모드 전환
resnet18.eval()


## 이미지 전처리: ResNet 모델 입력에 맞게 크기 조정 & 정규화
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


## 예제 이미지 불러오기(example.jpg 파일 필요)
img = Image.open("example.jpg").convert("RGB")
img_t = transform(img)
img_t = img_t.unsqueeze(0) # 배치 차원 추가

## 모델 예측
outputs = resnet18(img_t)
_, predicted = torch.max(outputs, 1)
print(f"예측된 이미지 클래스: {predicted.item()}")

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 85.0MB/s]


예측된 이미지 클래스: 1
